#### Copyright 2025, Battelle Energy Alliance, LLC All Rights Reserved

#### Import CyNER and get model

In [ ]:
import os, sys, json
import cyner
import spacy

import mlflow

#### CyNERWrapper

In [ ]:
import pathlib, mlflow, cyner, spacy
from typing import List, Dict, Any

class CyNERWrapper(mlflow.pyfunc.PythonModel):
    """
    MLflow pyfunc wrapper for cyner.CyNER that is portable across machines.
    """

    def __init__(self, cfg: Dict[str, Any]):
        self._raw_cfg = cfg
        self.model = None

    def load_context(self, context: mlflow.pyfunc.PythonModelContext):
        cfg = self._raw_cfg.copy()

        # Use MLflow-supplied artifact if present
        art_path = context.artifacts.get("transformers_ner")
        if art_path:
            cfg["transformers_model"] = str(pathlib.Path(art_path))

        # Instantiate the real CyNER model once
        self.model = cyner.CyNER(
            transformers_config=cfg,
            use_heuristic=cfg["use_heuristic"],
            flair_model   =cfg["flair_model"],
            priority      =cfg["priority"],
        )

        if cfg["finetune"]:
            print(f"Finetuning model on load: {cfg['transformers_model']}")
            self.model.transformer_ner.train()

    # ------------------------------------------------------------------ #
    def evaluate(self):
        return self.model.transformer_ner.get_test_eval()

    def predict(
        self, context: mlflow.pyfunc.PythonModelContext, model_input: List[str]
    ) -> List[Dict[str, Any]]:
        text = model_input[0]
        ents = self.model.get_entities(text)

        nlp = spacy.blank("en")
        doc = nlp(text)
        spans = [
            doc.char_span(e.start, e.end, label=e.entity_type, alignment_mode="expand")
            for e in ents
            if doc.char_span(e.start, e.end, label=e.entity_type)
        ]
        spans = spacy.util.filter_spans(spans)
        ents  = [e for e in ents if any(e.text == s.text for s in spans)]

        return [
            dict(
                entity_text=e.text,
                entity_label=e.entity_type,
                confidence=e.confidence,
                start_pos=e.start,
                end_pos=e.end,
            )
            for e in ents
        ]

In [3]:
import os, mlflow, pathlib

model_version = 'cyner-2_0-deberta-v3-base'

# ── pick a *relative* model folder name for portability
base_weights = '/data/models/CyNER-2.0-DeBERTa-v3-base'      # <- copied locally later
ckpt_dir     = f"/data/ckpts/{model_version}"          # fine-tune output

# model parameter configs
cfg = {
    'checkpoint_dir': ckpt_dir, # finetuned model output
    'transformers_model': base_weights, # base model to be further finetuned
    'dataset': '../dataset/train/mitre_cyner',
    'random_seed': 42,
    'lr': 3e-5,
    'epochs':25, #num_of_epochs,
    'warmup_step': 1000,
    'weight_decay': 0.01,
    'batch_size': 64,
    'max_seq_length': 128,
    'fp16': False,
    'max_grad_norm': 1.0,
    'lower_case': False,
    'num_worker': 0,
    'cache_dir': None,
    'use_heuristic': True, # use_regex,
    'flair_model': None, #'ner',
    'priority': 'HTFS', # priority of model results: Heuristic > Transformers > Flair > Spacy
    'finetune': False, # whether to finetune a pre-trained (or previously finetuned) model
}

In [4]:
tracking_uri = "/data/coreii_cyner/ner/mlruns"
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("MITRE-ATT&CK-ICS NER Experiments")

with mlflow.start_run() as run:
    model = CyNERWrapper(cfg)

    # choose which folder to log
    artifact_src = pathlib.Path(cfg["checkpoint_dir"] if cfg["finetune"]
                                else cfg["transformers_model"]).resolve()

    mlflow.pyfunc.log_model(
        artifact_path = model_version,
        python_model = model,
        artifacts = {"transformers_ner": str(artifact_src)},
        conda_env = None,
    )

    if cfg["finetune"]:
        mlflow.log_params(cfg)
        mlflow.log_metrics(model.evaluate())

    # Get the run ID for reference
    run_id = run.info.run_id
    print(f"CyNER model logged in run {run_id}\n")

2025/07/16 11:46:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/opt/conda/envs/py_3.12/lib/python3.12/site-packages/mlflow/types/type_hints.py:221: UserWarning: Any type hint is inferred as AnyType, and MLflow doesn't validate the data for this type. Please use a more specific type hint to enable data validation.
  dtype=Map(_infer_colspec_type_from_type_hint(type_hint=args[1]).dtype),


2025/07/16 11:46:53 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.21.0+4040d51) contains a local version label (+4040d51). MLflow logged a pip requirement for this package as 'torchvision==0.21.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2025/07/16 11:46:53 WARNING mlflow.utils.requirements_utils: Found amdsmi version (25.3.0+ede62f2) contains a local version label (+ede62f2). MLflow logged a pip requirement for this package as 'amdsmi==25.3.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


CyNER model logged in run 9834fe5d1f41451a97bd06b35c4c9129



#### Loading and Predicting with _Finetuned ToyNER_

In [5]:
# # Load the model using MLflow
logged_model_uri = f"runs:/{run_id}/{model_version}"
loaded_model = mlflow.pyfunc.load_model(logged_model_uri)

In [6]:
mlflow.pyfunc.get_model_dependencies(logged_model_uri)

2025/07/16 11:48:05 INFO mlflow.pyfunc: To install the dependencies that were used to train the model, run the following command: '%pip install -r /tmp/tmpywm9dz_i/cyner-2_0-deberta-v3-base/requirements.txt'.


'/tmp/tmpywm9dz_i/cyner-2_0-deberta-v3-base/requirements.txt'

In [7]:
def predict_entities (input_fp: str, model_version: str=model_version):
    # Open the file in read mode
    with open(input_fp, 'r') as file:
        # Read the entire content of the file
        text_str = file.read()
    
    # entities = loaded_model.predict(text_input)
    entities = loaded_model.predict([text_str])
    
    
    # # Write the data to a JSON file
    # fn = input_fp.split('/')[-1]
    # json_file_path = f"../output/cyner_{model_version}_{fn.split('.')[0]}.json"
    
    # with open(json_file_path, 'w') as json_file:
    #     json.dump(entities, json_file, indent=4)
    
    # print(f"Data successfully written to {json_file_path}")
    return entities

In [8]:
test_fn = 'AA20-302A.txt'
input_fp = f"../dataset/input/{test_fn}"

test1_output = predict_entities (input_fp)
test1_output

2025-07-16 11:48:16 INFO     *** initialize network ***


Using device: cuda


[{'entity_text': 'central@cisa.gov',
  'entity_label': 'item',
  'confidence': 1,
  'start_pos': 28858,
  'end_pos': 28874},
 {'entity_text': 'CyWatch@fbi.gov',
  'entity_label': 'item',
  'confidence': 1,
  'start_pos': 28919,
  'end_pos': 28934},
 {'entity_text': 'HC3@HHS.gov',
  'entity_label': 'item',
  'confidence': 1,
  'start_pos': 28968,
  'end_pos': 28979},
 {'entity_text': 'Central@cisa.gov',
  'entity_label': 'item',
  'confidence': 1,
  'start_pos': 38935,
  'end_pos': 38951},
 {'entity_text': 'CyWatch@fbi.gov',
  'entity_label': 'item',
  'confidence': 1,
  'start_pos': 39699,
  'end_pos': 39714},
 {'entity_text': 'Central@cisa.gov',
  'entity_label': 'item',
  'confidence': 1,
  'start_pos': 40117,
  'end_pos': 40133},
 {'entity_text': 'mfjdieks.exe',
  'entity_label': 'item',
  'confidence': 1,
  'start_pos': 4649,
  'end_pos': 4661},
 {'entity_text': 'anchorDiag.txt',
  'entity_label': 'item',
  'confidence': 1,
  'start_pos': 5898,
  'end_pos': 5912},
 {'entity_text': 

In [9]:
test_fn = 'TestPDFs.txt'
input_fp = f"../dataset/input/{test_fn}"
test2_output = predict_entities (input_fp)
test2_output

[{'entity_text': '/OT',
  'entity_label': 'location',
  'confidence': 1,
  'start_pos': 234,
  'end_pos': 237},
 {'entity_text': '/ICS',
  'entity_label': 'location',
  'confidence': 1,
  'start_pos': 10413,
  'end_pos': 10417},
 {'entity_text': 'Hanover, Maryland',
  'entity_label': 'location',
  'confidence': 1,
  'start_pos': 1137,
  'end_pos': 1154},
 {'entity_text': 'January 2024',
  'entity_label': 'date',
  'confidence': 1,
  'start_pos': 1726,
  'end_pos': 1738},
 {'entity_text': 'Tuesday',
  'entity_label': 'date',
  'confidence': 1,
  'start_pos': 679,
  'end_pos': 686},
 {'entity_text': 'Impact',
  'entity_label': 'tactic',
  'confidence': 1,
  'start_pos': 2046,
  'end_pos': 2052},
 {'entity_text': 'Connected OT Systems',
  'entity_label': 'org',
  'confidence': 1,
  'start_pos': 2082,
  'end_pos': 2102},
 {'entity_text': 'Forum',
  'entity_label': 'event',
  'confidence': 1,
  'start_pos': 10472,
  'end_pos': 10477},
 {'entity_text': 'Collaboration',
  'entity_label': 'eve

In [10]:
test_fn = 'Verizon Salt Typhoon cyberattack.txt'
input_fp = f"../dataset/input/{test_fn}"
test3_output = predict_entities (input_fp)
test3_output

[{'entity_text': 'American telecommunications company',
  'entity_label': 'org',
  'confidence': 0.8495475848515829,
  'start_pos': 27,
  'end_pos': 62},
 {'entity_text': 'cyber incident',
  'entity_label': 'item',
  'confidence': 0.756631463766098,
  'start_pos': 108,
  'end_pos': 122},
 {'entity_text': 'cybersecurity firm',
  'entity_label': 'org',
  'confidence': 0.8216371834278107,
  'start_pos': 173,
  'end_pos': 191},
 {'entity_text': 'The company',
  'entity_label': 'org',
  'confidence': 0.5170261114835739,
  'start_pos': 193,
  'end_pos': 204},
 {'entity_text': 'nation-state threat actor',
  'entity_label': 'org',
  'confidence': 0.8249266862869262,
  'start_pos': 232,
  'end_pos': 257},
 {'entity_text': 'hackers',
  'entity_label': 'org',
  'confidence': 0.9564035534858704,
  'start_pos': 301,
  'end_pos': 308},
 {'entity_text': 'breached',
  'entity_label': 'item',
  'confidence': 0.6371602416038513,
  'start_pos': 330,
  'end_pos': 338},
 {'entity_text': 'national',
  'enti

In [11]:
test_fn = 'WEC Fictional company profile.txt'
input_fp = f"../dataset/input/{test_fn}"
test3_output = predict_entities (input_fp)
test3_output

[{'entity_text': 'info@wecoilandgas.com',
  'entity_label': 'item',
  'confidence': 1,
  'start_pos': 2442,
  'end_pos': 2463},
 {'entity_text': 'Houston, TX',
  'entity_label': 'location',
  'confidence': 1,
  'start_pos': 2395,
  'end_pos': 2406},
 {'entity_text': 'Gas Company',
  'entity_label': 'org',
  'confidence': 1,
  'start_pos': 50,
  'end_pos': 61},
 {'entity_text': 'Gas Company',
  'entity_label': 'org',
  'confidence': 1,
  'start_pos': 1082,
  'end_pos': 1093},
 {'entity_text': 'Gas Company',
  'entity_label': 'org',
  'confidence': 1,
  'start_pos': 2097,
  'end_pos': 2108},
 {'entity_text': 'Gas Company',
  'entity_label': 'org',
  'confidence': 1,
  'start_pos': 2482,
  'end_pos': 2493},
 {'entity_text': '1924',
  'entity_label': 'date',
  'confidence': 0.6581038236618042,
  'start_pos': 29,
  'end_pos': 33},
 {'entity_text': 'energy provider',
  'entity_label': 'org',
  'confidence': 0.7923166453838348,
  'start_pos': 92,
  'end_pos': 107},
 {'entity_text': 'the Unite

#### Registering model for Deployment

In [12]:
# Register the model for deployment
model_name = "ToyCER_2.0"
model_details = mlflow.register_model(model_uri=logged_model_uri, name=model_name)

print(f"Model registered with name {model_name} and version {model_details.version}\n")

Successfully registered model 'ToyCER_2.0'.
2025/07/16 11:48:35 WARNING mlflow.tracking._model_registry.fluent: Run with id 9834fe5d1f41451a97bd06b35c4c9129 has no artifacts at artifact path 'cyner-2_0-deberta-v3-base', registering model based on models:/m-72ee04420f70427e8f2a9c269c8884e8 instead


Model registered with name ToyCER_2.0 and version 1



Created version '1' of model 'ToyCER_2.0'.
